In [0]:
%pip install pytest

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python or dbutils.library.restartPython()

In [0]:
import pytest
from pyspark.sql import functions as F


# ============================================================
# CATALOG
# ============================================================

CATALOG = "wsdbrksecommerce"


# ============================================================
# SILVER - CUSTOMER TESTS
# ============================================================

def test_customer_id_not_null():

    df = spark.table(
        f"{CATALOG}.silver.silver_customer_dataset"
    )

    result = df.filter(
        df.customer_id.isNull()
    ).count()

    assert result == 0, (
        f"Found {result} NULL customer_id values"
    )


def test_customer_id_unique():

    df = spark.table(
        f"{CATALOG}.silver.silver_customer_dataset"
    )

    result = (
        df.groupBy("customer_id")
        .count()
        .filter("count > 1")
        .count()
    )

    assert result == 0, (
        f"Found {result} duplicate customer_id groups"
    )


def test_customer_city_trimmed():

    df = spark.table(
        f"{CATALOG}.silver.silver_customer_dataset"
    )

    result = df.filter(
        df.customer_city != F.trim(df.customer_city)
    ).count()

    assert result == 0, (
        f"Found {result} untrimmed customer_city values"
    )


def test_customer_state_trimmed():

    df = spark.table(
        f"{CATALOG}.silver.silver_customer_dataset"
    )

    result = df.filter(
        df.customer_state != F.trim(df.customer_state)
    ).count()

    assert result == 0, (
        f"Found {result} untrimmed customer_state values"
    )


# ============================================================
# SILVER - ORDER TESTS
# ============================================================

def test_order_id_not_null():

    df = spark.table(
        f"{CATALOG}.silver.silver_orders_dataset"
    )

    result = df.filter(
        df.order_id.isNull()
    ).count()

    assert result == 0, (
        f"Found {result} NULL order_id values"
    )


def test_order_id_unique():

    df = spark.table(
        f"{CATALOG}.silver.silver_orders_dataset"
    )

    result = (
        df.groupBy("order_id")
        .count()
        .filter("count > 1")
        .count()
    )

    assert result == 0, (
        f"Found {result} duplicate order_id groups"
    )


def test_order_customer_reference():

    orders = spark.table(
        f"{CATALOG}.silver.silver_orders_dataset"
    )

    customers = spark.table(
        f"{CATALOG}.silver.silver_customer_dataset"
    )

    result = (
        orders
        .join(
            customers,
            orders.customer_id == customers.customer_id,
            "left"
        )
        .filter(
            customers.customer_id.isNull()
        )
        .count()
    )

    assert result == 0, (
        f"Found {result} orders without customer reference"
    )


def test_purchase_timestamp_not_null():

    df = spark.table(
        f"{CATALOG}.silver.silver_orders_dataset"
    )

    result = df.filter(
        df.order_purchase_timestamp.isNull()
    ).count()

    assert result == 0, (
        f"Found {result} NULL purchase timestamps"
    )


# ============================================================
# SILVER - ORDER TIMESTAMP LOGIC
# ============================================================

def test_order_timestamp_sequence():

    df = spark.table(
        f"{CATALOG}.silver.silver_orders_dataset"
    )

    result = df.filter(
        (
            df.order_approved_at.isNotNull()
        )
        &
        (
            df.order_approved_at
            < df.order_purchase_timestamp
        )
    ).count()

    assert result == 0, (
        f"Found {result} orders approved before purchase"
    )


def test_delivery_not_before_purchase():

    df = spark.table(
        f"{CATALOG}.silver.silver_orders_dataset"
    )

    result = df.filter(
        (
            df.order_delivered_customer_date.isNotNull()
        )
        &
        (
            df.order_delivered_customer_date
            < df.order_purchase_timestamp
        )
    ).count()

    assert result == 0, (
        f"Found {result} deliveries before purchase"
    )


def test_delivery_days_not_negative():

    df = spark.table(
        f"{CATALOG}.silver.silver_orders_dataset"
    )

    result = df.filter(
        df.delivery_days < 0
    ).count()

    assert result == 0, (
        f"Found {result} negative delivery_days"
    )


def test_delivered_orders_have_delivery_date():

    df = spark.table(
        f"{CATALOG}.silver.silver_orders_dataset"
    )

    result = df.filter(
        (
            df.order_status == "delivered"
        )
        &
        df.order_delivered_customer_date.isNull()
    ).count()

    assert result == 0, (
        f"Found {result} delivered orders without delivery date"
    )


# ============================================================
# SILVER - ORDER ITEMS TESTS
# ============================================================

def test_order_item_key_unique():

    df = spark.table(
        f"{CATALOG}.silver.silver_order_items_dataset"
    )

    result = (
        df.groupBy(
            "order_id",
            "order_item_id"
        )
        .count()
        .filter("count > 1")
        .count()
    )

    assert result == 0, (
        f"Found {result} duplicate order-item keys"
    )


def test_order_item_required_fields_not_null():

    df = spark.table(
        f"{CATALOG}.silver.silver_order_items_dataset"
    )

    result = df.filter(
        df.order_id.isNull()
        |
        df.order_item_id.isNull()
        |
        df.product_id.isNull()
        |
        df.seller_id.isNull()
    ).count()

    assert result == 0, (
        f"Found {result} order items with NULL required fields"
    )


def test_price_not_negative():

    df = spark.table(
        f"{CATALOG}.silver.silver_order_items_dataset"
    )

    result = df.filter(
        df.price < 0
    ).count()

    assert result == 0, (
        f"Found {result} negative price records"
    )


def test_freight_not_negative():

    df = spark.table(
        f"{CATALOG}.silver.silver_order_items_dataset"
    )

    result = df.filter(
        df.freight_value < 0
    ).count()

    assert result == 0, (
        f"Found {result} negative freight records"
    )


def test_total_item_value_calculation():

    df = spark.table(
        f"{CATALOG}.silver.silver_order_items_dataset"
    )

    result = df.filter(
        df.total_item_value
        != (
            df.price + df.freight_value
        )
    ).count()

    assert result == 0, (
        f"Found {result} incorrect total_item_value calculations"
    )


# ============================================================
# SILVER - PRODUCT TESTS
# ============================================================

def test_product_id_not_null():

    df = spark.table(
        f"{CATALOG}.silver.silver_products_dataset"
    )

    result = df.filter(
        df.product_id.isNull()
    ).count()

    assert result == 0, (
        f"Found {result} NULL product_id values"
    )


def test_product_id_unique():

    df = spark.table(
        f"{CATALOG}.silver.silver_products_dataset"
    )

    result = (
        df.groupBy("product_id")
        .count()
        .filter("count > 1")
        .count()
    )

    assert result == 0, (
        f"Found {result} duplicate product_id groups"
    )


def test_product_attributes_not_null():

    df = spark.table(
        f"{CATALOG}.silver.silver_products_dataset"
    )

    result = df.filter(
        df.product_name_length.isNull()
        |
        df.product_description_length.isNull()
        |
        df.product_photos_qty.isNull()
        |
        df.product_weight_g.isNull()
        |
        df.product_length_cm.isNull()
        |
        df.product_height_cm.isNull()
        |
        df.product_width_cm.isNull()
    ).count()

    assert result == 0, (
        f"Found {result} products with NULL attributes"
    )


def test_product_measurements_non_negative():

    df = spark.table(
        f"{CATALOG}.silver.silver_products_dataset"
    )

    result = df.filter(
        (df.product_weight_g < 0)
        |
        (df.product_length_cm < 0)
        |
        (df.product_height_cm < 0)
        |
        (df.product_width_cm < 0)
    ).count()

    assert result == 0, (
        f"Found {result} negative product measurements"
    )


# ============================================================
# SILVER - SELLER TESTS
# ============================================================

def test_seller_id_not_null():

    df = spark.table(
        f"{CATALOG}.silver.silver_sellers_dataset"
    )

    result = df.filter(
        df.seller_id.isNull()
    ).count()

    assert result == 0, (
        f"Found {result} NULL seller_id values"
    )


def test_seller_id_unique():

    df = spark.table(
        f"{CATALOG}.silver.silver_sellers_dataset"
    )

    result = (
        df.groupBy("seller_id")
        .count()
        .filter("count > 1")
        .count()
    )

    assert result == 0, (
        f"Found {result} duplicate seller_id groups"
    )


# ============================================================
# GOLD - FACT KEY TESTS
# ============================================================

def test_fact_order_item_key_not_null():

    df = spark.table(
        f"{CATALOG}.gold.fact_order_items"
    )

    result = df.filter(
        df.order_item_key.isNull()
    ).count()

    assert result == 0, (
        f"Found {result} NULL order_item_key values"
    )


def test_fact_order_item_key_unique():

    df = spark.table(
        f"{CATALOG}.gold.fact_order_items"
    )

    result = (
        df.groupBy("order_item_key")
        .count()
        .filter("count > 1")
        .count()
    )

    assert result == 0, (
        f"Found {result} duplicate order_item_key values"
    )


def test_fact_business_key_unique():

    df = spark.table(
        f"{CATALOG}.gold.fact_order_items"
    )

    result = (
        df.groupBy(
            "order_id",
            "order_item_id"
        )
        .count()
        .filter("count > 1")
        .count()
    )

    assert result == 0, (
        f"Found {result} duplicate fact business keys"
    )


# ============================================================
# GOLD - DIMENSION RELATIONSHIPS
# ============================================================

def test_fact_customer_relationship():

    fact = spark.table(
        f"{CATALOG}.gold.fact_order_items"
    )

    customer = spark.table(
        f"{CATALOG}.gold.dim_customer"
    )

    result = (
        fact
        .join(
            customer,
            fact.customer_key == customer.customer_key,
            "left"
        )
        .filter(
            customer.customer_key.isNull()
        )
        .count()
    )

    assert result == 0, (
        f"Found {result} fact records without customer dimension"
    )


def test_fact_product_relationship():

    fact = spark.table(
        f"{CATALOG}.gold.fact_order_items"
    )

    product = spark.table(
        f"{CATALOG}.gold.dim_product"
    )

    result = (
        fact
        .join(
            product,
            fact.product_key == product.product_key,
            "left"
        )
        .filter(
            product.product_key.isNull()
        )
        .count()
    )

    assert result == 0, (
        f"Found {result} fact records without product dimension"
    )


def test_fact_seller_relationship():

    fact = spark.table(
        f"{CATALOG}.gold.fact_order_items"
    )

    seller = spark.table(
        f"{CATALOG}.gold.dim_seller"
    )

    result = (
        fact
        .join(
            seller,
            fact.seller_key == seller.seller_key,
            "left"
        )
        .filter(
            seller.seller_key.isNull()
        )
        .count()
    )

    assert result == 0, (
        f"Found {result} fact records without seller dimension"
    )


def test_fact_order_relationship():

    fact = spark.table(
        f"{CATALOG}.gold.fact_order_items"
    )

    orders = spark.table(
        f"{CATALOG}.gold.dim_order"
    )

    result = (
        fact
        .join(
            orders,
            fact.order_key == orders.order_key,
            "left"
        )
        .filter(
            orders.order_key.isNull()
        )
        .count()
    )

    assert result == 0, (
        f"Found {result} fact records without order dimension"
    )


def test_fact_category_relationship():

    fact = spark.table(
        f"{CATALOG}.gold.fact_order_items"
    )

    category = spark.table(
        f"{CATALOG}.gold.dim_product_category"
    )

    result = (
        fact
        .join(
            category,
            fact.product_category_key
            == category.product_category_key,
            "left"
        )
        .filter(
            category.product_category_key.isNull()
        )
        .count()
    )

    assert result == 0, (
        f"Found {result} fact records without category dimension"
    )


# ============================================================
# GOLD - SALES VALIDATION
# ============================================================

def test_fact_price_not_negative():

    df = spark.table(
        f"{CATALOG}.gold.fact_order_items"
    )

    result = df.filter(
        df.price < 0
    ).count()

    assert result == 0, (
        f"Found {result} negative price records"
    )


def test_fact_freight_not_negative():

    df = spark.table(
        f"{CATALOG}.gold.fact_order_items"
    )

    result = df.filter(
        df.freight_value < 0
    ).count()

    assert result == 0, (
        f"Found {result} negative freight records"
    )


def test_fact_quantity_positive():

    df = spark.table(
        f"{CATALOG}.gold.fact_order_items"
    )

    result = df.filter(
        df.quantity <= 0
    ).count()

    assert result == 0, (
        f"Found {result} invalid quantity records"
    )


def test_fact_total_item_value():

    df = spark.table(
        f"{CATALOG}.gold.fact_order_items"
    )

    result = df.filter(
        df.total_item_value
        != (
            df.price + df.freight_value
        )
    ).count()

    assert result == 0, (
        f"Found {result} incorrect total_item_value records"
    )


# ============================================================
# GOLD - DELIVERY VALIDATION
# ============================================================

def test_fact_delivery_days_not_negative():

    df = spark.table(
        f"{CATALOG}.gold.fact_order_items"
    )

    result = df.filter(
        df.delivery_days < 0
    ).count()

    assert result == 0, (
        f"Found {result} negative delivery_days"
    )


def test_fact_delivery_status_valid():

    df = spark.table(
        f"{CATALOG}.gold.fact_order_items"
    )

    valid_statuses = [
        "Late",
        "On-Time",
        "Unknown"
    ]

    result = df.filter(
        ~df.delivery_status.isin(valid_statuses)
    ).count()

    assert result == 0, (
        f"Found {result} invalid delivery statuses"
    )


def test_late_delivery_logic():

    df = spark.table(
        f"{CATALOG}.gold.fact_order_items"
    )

    result = df.filter(
        (df.delivery_status == "Late")
        &
        (
            df.order_delivered_customer_date
            <= df.order_estimated_delivery_date
        )
    ).count()

    assert result == 0, (
        f"Found {result} incorrectly classified late deliveries"
    )


def test_on_time_delivery_logic():

    df = spark.table(
        f"{CATALOG}.gold.fact_order_items"
    )

    result = df.filter(
        (df.delivery_status == "On-Time")
        &
        (df.delivery_delay_days > 0)
    ).count()

    assert result == 0, (
        f"Found {result} incorrectly classified on-time deliveries"
    )


# ============================================================
# GOLD - DATA QUALITY FLAGS
# ============================================================

def test_missing_customer_flag():

    df = spark.table(
        f"{CATALOG}.gold.fact_order_items"
    )

    result = df.filter(
        df.missing_customer_flag == 1
    ).count()

    assert result == 0, (
        f"Found {result} missing customer records"
    )


def test_missing_product_flag():

    df = spark.table(
        f"{CATALOG}.gold.fact_order_items"
    )

    result = df.filter(
        df.missing_product_flag == 1
    ).count()

    assert result == 0, (
        f"Found {result} missing product records"
    )


def test_missing_seller_flag():

    df = spark.table(
        f"{CATALOG}.gold.fact_order_items"
    )

    result = df.filter(
        df.missing_seller_flag == 1
    ).count()

    assert result == 0, (
        f"Found {result} missing seller records"
    )


def test_missing_category_flag():

    df = spark.table(
        f"{CATALOG}.gold.fact_order_items"
    )

    result = df.filter(
        df.missing_category_flag == 1
    ).count()

    assert result == 0, (
        f"Found {result} missing category records"
    )


# ============================================================
# RUN ALL TESTS
# ============================================================

tests = [

    # Silver - Customer
    test_customer_id_not_null,
    test_customer_id_unique,
    test_customer_city_trimmed,
    test_customer_state_trimmed,

    # Silver - Orders
    test_order_id_not_null,
    test_order_id_unique,
    test_order_customer_reference,
    test_purchase_timestamp_not_null,
    test_order_timestamp_sequence,
    test_delivery_not_before_purchase,
    test_delivery_days_not_negative,
    test_delivered_orders_have_delivery_date,

    # Silver - Order Items
    test_order_item_key_unique,
    test_order_item_required_fields_not_null,
    test_price_not_negative,
    test_freight_not_negative,
    test_total_item_value_calculation,

    # Silver - Products
    test_product_id_not_null,
    test_product_id_unique,
    test_product_attributes_not_null,
    test_product_measurements_non_negative,

    # Silver - Sellers
    test_seller_id_not_null,
    test_seller_id_unique,

    # Gold - Fact
    test_fact_order_item_key_not_null,
    test_fact_order_item_key_unique,
    test_fact_business_key_unique,

    # Gold - Relationships
    test_fact_customer_relationship,
    test_fact_product_relationship,
    test_fact_seller_relationship,
    test_fact_order_relationship,
    test_fact_category_relationship,

    # Gold - Sales
    test_fact_price_not_negative,
    test_fact_freight_not_negative,
    test_fact_quantity_positive,
    test_fact_total_item_value,

    # Gold - Delivery
    test_fact_delivery_days_not_negative,
    test_fact_delivery_status_valid,
    test_late_delivery_logic,
    test_on_time_delivery_logic,

    # Gold - Data Quality Flags
    test_missing_customer_flag,
    test_missing_product_flag,
    test_missing_seller_flag,
    test_missing_category_flag,
]


# ============================================================
# EXECUTE TESTS
# ============================================================

passed = 0
failed = 0

print("\n==========================================")
print("E-COMMERCE DATA QUALITY TESTS")
print("==========================================\n")

for test in tests:

    try:

        test()

        print(
            f"PASSED: {test.__name__}"
        )

        passed += 1

    except Exception as e:

        print(
            f"FAILED: {test.__name__}"
        )

        print(
            f"       {e}"
        )

        failed += 1


# ============================================================
# FINAL RESULT
# ============================================================

print("\n==========================================")
print(f"TOTAL TESTS : {len(tests)}")
print(f"PASSED      : {passed}")
print(f"FAILED      : {failed}")
print("==========================================")

if failed == 0:

    print("\nALL DATA QUALITY TESTS PASSED")

else:

    print("\nSOME DATA QUALITY TESTS FAILED")


E-COMMERCE DATA QUALITY TESTS

PASSED: test_customer_id_not_null
PASSED: test_customer_id_unique
PASSED: test_customer_city_trimmed
PASSED: test_customer_state_trimmed
PASSED: test_order_id_not_null
PASSED: test_order_id_unique
PASSED: test_order_customer_reference
PASSED: test_purchase_timestamp_not_null
PASSED: test_order_timestamp_sequence
PASSED: test_delivery_not_before_purchase
PASSED: test_delivery_days_not_negative
FAILED: test_delivered_orders_have_delivery_date
       Found 8 delivered orders without delivery date
PASSED: test_order_item_key_unique
PASSED: test_order_item_required_fields_not_null
PASSED: test_price_not_negative
PASSED: test_freight_not_negative
PASSED: test_total_item_value_calculation
PASSED: test_product_id_not_null
PASSED: test_product_id_unique
PASSED: test_product_attributes_not_null
PASSED: test_product_measurements_non_negative
PASSED: test_seller_id_not_null
PASSED: test_seller_id_unique
PASSED: test_fact_order_item_key_not_null
PASSED: test_fact_ord